In [1]:
import os 

In [2]:
%pwd

'/Users/nadeeramunasinghe/Desktop/ML projects/text summary project/text-summariser-project-/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/Users/nadeeramunasinghe/Desktop/ML projects/text summary project/text-summariser-project-'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: Path




In [6]:
from TextSummarizer.constants import*
from TextSummarizer.utils.common import read_yaml,create_directories


In [11]:
class configurationManager:
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
    
    def get_data_transformation_config(self) ->DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir = config.root_dir,
            data_path = config.data_path,
            tokenizer_name = config.tokenizer_name
        )

        return data_transformation_config

        

In [12]:
import os 
from TextSummarizer.logging import logger
from transformers import AutoTokenizer
from datasets import load_dataset, load_from_disk

In [15]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)

    def convert_examples_to_features(self, example_batch):
        input_encoding = self.tokenizer(example_batch['dialogue'], max_length = 1024, truncation = True)

        with self.tokenizer.as_target_tokenizer():
            target_encoding = self.tokenizer(example_batch['summary'], max_length = 1024, truncation=True)
        return{
            'input_ids' : input_encoding['input_ids'],
            'attention_mask' : input_encoding['attention_mask'],
            'labels': target_encoding['input_ids']

        }
    def convert(self):
        dataset_samsum = load_from_disk(self.config.data_path)
        dataset_samsum_pt = dataset_samsum.map(self.convert_examples_to_features, batch_size=True)
        dataset_samsum_pt.save_to_disk(os.path.join(self.config.root_dir,"samsum_dataset"))
        

In [16]:
try:
    config = configurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.convert()

except Exception as e:
    raise e 


[2026-07-02 16:14:35,614: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-07-02 16:14:35,616: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-02 16:14:35,616: INFO: common: created directory at: artifacts]
[2026-07-02 16:14:35,617: INFO: common: created directory at: artifacts/data_transformation]


'(ProtocolError('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer')), '(Request ID: c138cec0-381e-4310-898c-8259b17bc1e9)')' thrown while requesting HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json


[2026-07-02 16:14:35,633: WARNING: _http: '(ProtocolError('Connection aborted.', ConnectionResetError(54, 'Connection reset by peer')), '(Request ID: c138cec0-381e-4310-898c-8259b17bc1e9)')' thrown while requesting HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json]


Retrying in 1s [Retry 1/5].


[2026-07-02 16:14:35,634: WARNING: _http: Retrying in 1s [Retry 1/5].]


Map:   0%|          | 0/14732 [00:00<?, ? examples/s]/opt/anaconda3/envs/textSummary/lib/python3.8/site-packages/transformers/tokenization_utils_base.py:4114: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 422685.80 examples/s]
